# PSG-Audio — second téléchargement depuis Kaggle

Fichier : `notebooks/psg_audio_download_kaggle.ipynb`. Accélère le téléchargement de PSG-Audio
(ScienceDB, CC BY 4.0, données publiques) en le faisant tourner depuis un autre réseau que le PC.

- La liste figée (graine 42) est parcourue **depuis la fin** (`--reverse`) : le PC la parcourt depuis
  le début, les deux ne se croisent qu'à la fin.
- Arrêt avant **8 Go** d'EDF (`--max-gb 8`) et avant **11 h** de calcul (limite Kaggle : 12 h).
- Seules les voies non audio sont récupérées ; chaque `.rml` est contrôlé par son MD5 ; l'empreinte
  SHA-256 de chaque EDF est notée dans `download_log.json`, recontrôlée sur le PC avant intégration.

**Lancement** : Settings > Internet **on** (accélérateur : aucun), puis **Save Version > Save & Run All**.
À la fin, onglet *Output* : télécharger `psg_audio_kaggle.zip`.


In [ ]:
import os
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
%cd {WORK}
REVISION = "origin/main"
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
!df -h {WORK} | tail -1

In [ ]:
OUT = f"{WORK}/psg_audio"
MAX_GB = 8
# timeout : arrêt propre avant la limite de 12 h de Kaggle (journal écrit après chaque patient)
!cd {WORK}/sleepfm-pipeline && timeout 39600 python -u scripts/download_psg_audio.py --out {OUT} --reverse --max-gb {MAX_GB} ; echo "code de sortie $?" 

In [ ]:
import glob, json, zipfile
for part in glob.glob(f"{OUT}/*.part"):
    os.remove(part)  # patient interrompu par le timeout : refait sur le PC
log = json.load(open(f"{OUT}/download_log.json", encoding="utf-8"))
edfs = sorted(glob.glob(f"{OUT}/*.edf"))
print(f"{len(log['done'])} patients faits, {len(log['failed'])} en échec : {log['failed']}")
print(f"{sum(os.path.getsize(p) for p in edfs) / 1e9:.2f} Go d'EDF")
with zipfile.ZipFile(f"{WORK}/psg_audio_kaggle.zip", "w", zipfile.ZIP_STORED) as archive:
    for path in sorted(glob.glob(f"{OUT}/*")):
        archive.write(path, os.path.basename(path))
for path in glob.glob(f"{OUT}/*"):
    os.remove(path)  # seul le zip reste dans les sorties
!rm -rf {WORK}/sleepfm-pipeline
!ls -la {WORK}